# BOOKNOW — walkthrough

A guided tour of the library in `src/booknow/`. Every cell below imports real, tested modules.

```bash
pip install -e ".[ui]"
cp .env.example .env   # then paste your key
```


In [ ]:
from booknow import load_default_catalog

catalog = load_default_catalog()
print(len(catalog), "books |", ", ".join(catalog.genres()))

## 1. Lookup is fuzzy

A model paraphrases titles constantly. Exact dictionary lookup misses most of these;
all of them resolve here.

In [ ]:
for q in ["Atomic Habits", "atomic habits", "atomic habbits", "Silent Patient", "1984"]:
    book = catalog.find(q)
    print(f"{q!r:22} -> {book.title} ({book.price}, stock {book.stock})")

In [ ]:
from booknow.catalog import BookNotFoundError

try:
    catalog.find("The Hobbit")
except BookNotFoundError as exc:
    print("did you mean:", exc.suggestions)

## 2. Tools are a registry

Schemas sent to the API and the dispatcher are both generated from it.

In [ ]:
from booknow.tools import REGISTRY, dispatch, tool_schemas

print(sorted(REGISTRY))
tool_schemas()[0]

In [ ]:
dispatch("get_price", {"title": "thinking fast and slow"}, catalog)

In [ ]:
# Errors come back as data, so a bad call never kills the conversation
dispatch("order_pizza", {}, catalog)

## 3. The agent

Needs a real key from here on.

In [ ]:
from booknow import BookstoreAgent, Settings
from booknow.agent import build_client

settings = Settings.from_env()
agent = BookstoreAgent(client=build_client(settings), catalog=catalog, settings=settings)

print(agent.reply("how much is atomic habits, and is the silent patient in stock?"))
print("tools used:", agent.tool_calls_made)

## 4. The UI

In [ ]:
from booknow.app import build_demo

build_demo().launch()

## 5. Testing it without spending anything

The client is injected, so tests script the model's responses instead of calling it.
See `tests/conftest.py`.

In [ ]:
import sys

sys.path.insert(0, "../tests")

from conftest import FakeClient, text_response, tool_response  # noqa: E402

client = FakeClient(scripted=[
    tool_response(("get_price", {"title": "atomic habits"})),
    text_response("Atomic Habits is $18.75."),
])
offline_agent = BookstoreAgent(client=client, catalog=catalog)
print(offline_agent.reply("price?"), "| tools:", offline_agent.tool_calls_made)